# Hydrogeological and Geospatial Data for Surface and Subsurface Flow Modelling in Swedish Coastal Catchments Exploration with `mlcroissant`

This notebook demonstrates how to load, inspect, and explore a dataset defined by a Croissant schema using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via the following Croissant schema URL:

- `https://sen.science/doi/10.71728/senscience.8kx9-mx2j/fair2.json`

### Notes
- All dataset entities should be referenced by their `@id` values when available.
- This dataset package exposes rich metadata, but the provided package summary indicates that `recordSet` is currently empty. Therefore, this notebook includes robust inspection steps and conditional code paths so it remains useful whether tabular record sets are present at runtime or not.
- The notebook uses variables throughout so that paths and IDs can be updated dynamically.

In [ ]:
# Ensure required libraries are installed
!pip install -q mlcroissant pandas matplotlib seaborn

## 1. Data Loading

We start by loading the dataset from its Croissant JSON-LD URL. We then inspect the metadata object safely, avoiding dictionary-style access on `dataset.metadata` itself.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
from pprint import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.8kx9-mx2j/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Convert metadata to a plain JSON-like object for convenient inspection
metadata_json = dataset.metadata.to_json()

# Print a short summary from the JSON representation
print('Name:', metadata_json.get('name'))
print('Identifier:', metadata_json.get('identifier'))
print('Version:', metadata_json.get('version'))
print('License:', metadata_json.get('license'))
print('\nDescription:')
print(metadata_json.get('description'))

## 2. Metadata Overview

This section explores the dataset-level metadata, including high-level descriptive fields and any declared record sets.

Because Croissant datasets can vary in structure, we inspect the metadata programmatically and collect IDs where available.

In [ ]:
# Show the top-level metadata keys available in the JSON form
top_level_keys = sorted(metadata_json.keys())
print('Top-level metadata keys:')
print(top_level_keys)

In [ ]:
# Display selected dataset-level metadata fields
summary_fields = [
    '@id', 'name', 'identifier', 'version', 'datePublished',
    'spatialCoverage', 'temporalCoverage', 'license', 'citeAs',
    'keywords', 'dataCollectionType', 'dataUseCases', 'dataLimitations'
]

summary = {field: metadata_json.get(field) for field in summary_fields}
pprint(summary)

### Inspect declared record sets

Croissant data access typically revolves around `recordSet` definitions. We inspect them here and extract their `@id` values.

If no record sets are present, the notebook will continue with metadata-focused analysis.

In [ ]:
record_sets_metadata = metadata_json.get('recordSet', []) or []
print(f'Number of declared record sets: {len(record_sets_metadata)}')

record_set_ids = []
for i, record_set in enumerate(record_sets_metadata):
    record_set_id = record_set.get('@id', f'<missing-record-set-id-{i}>') if isinstance(record_set, dict) else f'<non-dict-record-set-{i}>'
    record_set_name = record_set.get('name') if isinstance(record_set, dict) else None
    record_set_description = record_set.get('description') if isinstance(record_set, dict) else None
    record_set_ids.append(record_set_id)
    print(f'\nRecord set {i + 1}')
    print('  @id:', record_set_id)
    print('  name:', record_set_name)
    print('  description:', record_set_description)

record_set_ids

## 3. Data Overview

We now attempt to inspect records from each record set using its `@id`.

The code below is defensive:
- if record sets exist, it prints a small sample of records;
- if they do not, it clearly reports that no record-level extraction is available from the current metadata.

In [ ]:
sample_records = {}

if not record_set_ids:
    print('No record sets were declared in the dataset metadata. Skipping record preview.')
else:
    for record_set_id in record_set_ids:
        print(f'\nSampling records from record set: {record_set_id}')
        try:
            rows = []
            for idx, record in enumerate(dataset.records(record_set=record_set_id)):
                rows.append(record)
                print(record)
                if idx >= 2:
                    break
            sample_records[record_set_id] = rows
            if not rows:
                print('  No records returned.')
        except Exception as e:
            print(f'  Could not load records for {record_set_id}: {e}')

## 4. Data Extraction

In this section, we load each available record set into a pandas DataFrame.

As recommended, all record sets are handled through variables, and references use their `@id` values.

In [ ]:
dataframes = {}

if not record_set_ids:
    print('No record sets available, so no DataFrames can be created from records.')
else:
    for record_set_id in record_set_ids:
        try:
            records = list(dataset.records(record_set=record_set_id))
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f'{record_set_id}: shape={df.shape}')
        except Exception as e:
            print(f'Failed to create DataFrame for {record_set_id}: {e}')

list(dataframes.keys())

In [ ]:
# Display columns and a preview for the first non-empty DataFrame, if one exists
non_empty_record_set_id = None
for rs_id, df in dataframes.items():
    if not df.empty:
        non_empty_record_set_id = rs_id
        break

if non_empty_record_set_id is None:
    print('No non-empty DataFrame is available for preview.')
else:
    print('Selected record set @id:', non_empty_record_set_id)
    print('Columns:')
    print(dataframes[non_empty_record_set_id].columns.tolist())
    display(dataframes[non_empty_record_set_id].head())

## 5. Exploratory Data Analysis (EDA)

This section follows the template style and performs common preprocessing steps when tabular records are available:
- identify numeric fields,
- filter records using a threshold,
- normalize a numeric field,
- optionally group by a categorical field.

If no record sets are available, we instead perform a compact metadata-oriented EDA on dataset-level fields such as keywords, data collection types, and stated limitations.

In [ ]:
import numpy as np

if non_empty_record_set_id is not None:
    eda_df = dataframes[non_empty_record_set_id].copy()
    
    # Attempt to identify numeric fields dynamically
    numeric_columns = eda_df.select_dtypes(include='number').columns.tolist()
    
    # If numeric types are not inferred yet, try a soft conversion for object columns
    if not numeric_columns:
        converted_df = eda_df.copy()
        for col in converted_df.columns:
            converted_df[col] = pd.to_numeric(converted_df[col], errors='ignore')
        numeric_columns = converted_df.select_dtypes(include='number').columns.tolist()
        eda_df = converted_df
    
    print('Numeric columns detected:', numeric_columns)
    
    if numeric_columns:
        numeric_field = numeric_columns[0]
        threshold = eda_df[numeric_field].dropna().median()
        filtered_df = eda_df[eda_df[numeric_field] > threshold].copy()
        
        print(f'Using numeric field: {numeric_field}')
        print(f'Filtering records where {numeric_field} > {threshold}')
        display(filtered_df.head())
        
        std_value = filtered_df[numeric_field].std()
        if pd.notna(std_value) and std_value != 0:
            filtered_df[f'{numeric_field}_normalized'] = (
                (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / std_value
            )
        else:
            filtered_df[f'{numeric_field}_normalized'] = 0.0
        
        print(f'Normalized field: {numeric_field}_normalized')
        display(filtered_df[[numeric_field, f'{numeric_field}_normalized']].head())
        
        candidate_group_fields = [
            col for col in filtered_df.columns
            if col != numeric_field and filtered_df[col].dtype == 'object'
        ]
        
        if candidate_group_fields:
            group_field = candidate_group_fields[0]
            print(f'Grouping by field: {group_field}')
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)
            display(grouped_df.head())
        else:
            print('No suitable categorical group field found for grouping.')
    else:
        print('No numeric fields found in the selected record set. EDA is limited for record-level analysis.')
else:
    print('No record-level tables available. Performing metadata-oriented EDA instead.\n')
    
    keywords = metadata_json.get('keywords', []) or []
    data_collection_types = metadata_json.get('dataCollectionType', []) or []
    data_use_cases = metadata_json.get('dataUseCases', []) or []
    data_limitations = metadata_json.get('dataLimitations', []) or []
    
    metadata_eda = pd.DataFrame({
        'metadata_field': ['keywords', 'dataCollectionType', 'dataUseCases', 'dataLimitations'],
        'count': [len(keywords), len(data_collection_types), len(data_use_cases), len(data_limitations)]
    })
    display(metadata_eda)
    
    print('Keywords:')
    print(keywords)
    print('\nData collection types:')
    print(data_collection_types)
    print('\nData use cases:')
    print(data_use_cases)
    print('\nData limitations:')
    print(data_limitations)

## 6. Visualization

We create simple visualizations depending on what is available:
- for record-level data: histogram of a numeric field and, if possible, a grouped bar chart;
- otherwise: a metadata summary bar chart and keyword frequency view.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

if non_empty_record_set_id is not None and 'eda_df' in globals() and 'numeric_columns' in globals() and numeric_columns:
    plot_numeric_field = numeric_columns[0]
    
    fig, ax = plt.subplots(1, 1, figsize=(8, 5))
    sns.histplot(eda_df[plot_numeric_field].dropna(), bins=30, kde=True, ax=ax)
    ax.set_title(f'Distribution of {plot_numeric_field}\nRecord set: {non_empty_record_set_id}')
    ax.set_xlabel(plot_numeric_field)
    plt.tight_layout()
    plt.show()
    
    if 'candidate_group_fields' in globals() and candidate_group_fields:
        plot_group_field = candidate_group_fields[0]
        top_counts = (
            eda_df[plot_group_field]
            .astype(str)
            .value_counts()
            .head(10)
            .reset_index()
        )
        top_counts.columns = [plot_group_field, 'count']
        
        fig, ax = plt.subplots(1, 1, figsize=(10, 5))
        sns.barplot(data=top_counts, x=plot_group_field, y='count', ax=ax)
        ax.set_title(f'Top categories in {plot_group_field}')
        ax.tick_params(axis='x', rotation=45)
        plt.tight_layout()
        plt.show()
else:
    keywords = metadata_json.get('keywords', []) or []
    data_collection_types = metadata_json.get('dataCollectionType', []) or []
    data_use_cases = metadata_json.get('dataUseCases', []) or []
    data_limitations = metadata_json.get('dataLimitations', []) or []
    
    counts_df = pd.DataFrame({
        'category': ['keywords', 'dataCollectionType', 'dataUseCases', 'dataLimitations'],
        'count': [len(keywords), len(data_collection_types), len(data_use_cases), len(data_limitations)]
    })
    
    fig, ax = plt.subplots(1, 1, figsize=(8, 5))
    sns.barplot(data=counts_df, x='category', y='count', ax=ax)
    ax.set_title('Counts of Selected Metadata Collections')
    ax.tick_params(axis='x', rotation=20)
    plt.tight_layout()
    plt.show()
    
    if keywords:
        keyword_df = pd.DataFrame({'keyword': keywords, 'count': [1] * len(keywords)})
        fig, ax = plt.subplots(1, 1, figsize=(10, 5))
        sns.barplot(data=keyword_df, x='keyword', y='count', ax=ax, color='steelblue')
        ax.set_title('Dataset Keywords')
        ax.tick_params(axis='x', rotation=45)
        plt.tight_layout()
        plt.show()

## 7. Additional Metadata Inspection by `@id`

To support Croissant-style exploration, the following cell extracts selected linked entities that are already expressed as `@id` references in the dataset metadata, such as authors, citations, and distributions.

In [ ]:
def extract_id_list(items):
    ids = []
    for item in items or []:
        if isinstance(item, dict) and '@id' in item:
            ids.append(item['@id'])
    return ids

author_ids = extract_id_list(metadata_json.get('author', []))
citation_ids = extract_id_list(metadata_json.get('citation', []))
distribution_ids = extract_id_list(metadata_json.get('distribution', []))
visualization_ids = extract_id_list(metadata_json.get('visualization', []))

linked_entities_df = pd.DataFrame({
    'entity_type': ['author', 'citation', 'distribution', 'visualization'],
    'count': [len(author_ids), len(citation_ids), len(distribution_ids), len(visualization_ids)]
})

display(linked_entities_df)

print('Author @id values:')
print(author_ids)
print('\nCitation @id values:')
print(citation_ids[:10], '...' if len(citation_ids) > 10 else '')
print('\nDistribution @id values:')
print(distribution_ids[:10], '...' if len(distribution_ids) > 10 else '')
print('\nVisualization @id values:')
print(visualization_ids)

## 8. Conclusion

This notebook loaded the dataset from its Croissant schema and inspected its metadata using `mlcroissant`.

### Key observations
- The dataset describes hydrogeological and geospatial resources for Swedish coastal catchment modelling.
- The metadata includes rich descriptive information such as keywords, use cases, limitations, spatial coverage, and linked entities by `@id`.
- In the provided package summary, `recordSet` appears to be empty, so record-level extraction may not be available unless the runtime metadata resolves additional tabular definitions.
- The notebook is written defensively, so if record sets are added or resolved later, the same workflow can be reused for loading records, creating DataFrames, and performing EDA.

### Next steps
- Inspect the full Croissant document and any linked distributions to determine whether record sets are defined externally.
- If tabular or geospatial records become available through Croissant record sets, replace the automatically selected analysis fields with the specific field `@id` values you want to study.
- Extend this workflow with domain-specific spatial analysis tools such as `geopandas`, `rasterio`, or `xarray` where appropriate.